In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
import pypots
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.font_manager import FontProperties
plt.rcParams['font.sans-serif'] = ['SimHei']  
plt.rcParams['axes.unicode_minus'] = False  

# 定义基础路径（使用相对路径）
BASE_DIR = os.path.dirname(os.path.abspath(''))
if not BASE_DIR:
    BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "pipotsenvironmentallaw", "data", "air_quality")
RESULTS_DIR = os.path.join(BASE_DIR, "pipotsenvironmentallaw", "results", "imputed")
PLOTS_DIR = os.path.join(BASE_DIR, "pipotsenvironmentallaw", "results", "plots")

class Visualizer:
    def __init__(self, plots_dir):
        """初始化可视化器"""
        self.plots_dir = plots_dir
        os.makedirs(plots_dir, exist_ok=True)
        
    def plot_data_distribution(self, df, feature_columns, title):
        """绘制数据分布图"""
        plt.figure(figsize=(15, 5))
        for i, col in enumerate(feature_columns, 1):
            plt.subplot(1, 3, i)
            sns.histplot(data=df[col], kde=True)
            plt.title(f'{col}分布')
            plt.xlabel('值')
            plt.ylabel('频次')
        plt.tight_layout()
        plt.savefig(os.path.join(self.plots_dir, f'{title}_distribution.png'))
        plt.close()
        
    def plot_time_series(self, original, imputed, mask, feature_name, station_id):
        """绘制时间序列对比图"""
        plt.figure(figsize=(12, 6))
        time_points = range(len(original))
        
        # 绘制原始数据
        plt.plot(time_points, original, 'b-', label='原始数据', alpha=0.7)
        
        # 绘制缺失点
        missing_points = np.where(mask)[0]
        plt.scatter(missing_points, imputed[missing_points], 
                   color='r', label='插补数据', alpha=0.7)
        
        plt.title(f'站点{station_id} - {feature_name}时间序列插补对比')
        plt.xlabel('时间步')
        plt.ylabel('值')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(self.plots_dir, 
                                f'timeseries_{feature_name}_station_{station_id}.png'))
        plt.close()
        
    def plot_correlation_matrix(self, df, title):
        """绘制相关性矩阵"""
        plt.figure(figsize=(8, 6))
        sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0)
        plt.title(f'{title}特征相关性矩阵')
        plt.savefig(os.path.join(self.plots_dir, f'{title}_correlation.png'))
        plt.close()
        
    def plot_boxplot(self, df, feature_columns, title):
        """绘制箱线图"""
        plt.figure(figsize=(12, 6))
        df[feature_columns].boxplot()
        plt.title(f'{title}箱线图')
        plt.xticks(rotation=45)
        plt.ylabel('值')
        plt.tight_layout()
        plt.savefig(os.path.join(self.plots_dir, f'{title}_boxplot.png'))
        plt.close()
        
    def plot_imputation_comparison(self, original_data, imputed_data, mask, 
                                 feature_columns, station_ids):
        """绘制插补效果对比图"""
        for station_id in station_ids[:3]:  # 选择前3个站点进行展示
            for feature_idx, feature_name in enumerate(feature_columns):
                original = original_data[station_id, :, feature_idx]
                imputed = imputed_data[station_id, :, feature_idx]
                station_mask = mask[station_id, :, feature_idx]
                self.plot_time_series(original, imputed, station_mask, 
                                    feature_name, station_id)
    
    def plot_metrics_comparison(self, metrics_dict, title):
        """绘制评估指标对比图"""
        plt.figure(figsize=(10, 6))
        metrics = pd.DataFrame(metrics_dict)
        metrics.plot(kind='bar')
        plt.title(f'{title}评估指标对比')
        plt.xlabel('指标类型')
        plt.ylabel('值')
        plt.legend(title='数据集')
        plt.tight_layout()
        plt.savefig(os.path.join(self.plots_dir, f'{title}_metrics_comparison.png'))
        plt.close()

class ResultsSaver:
    def __init__(self, results_dir, plots_dir):
        """初始化结果保存器"""
        self.results_dir = results_dir
        self.visualizer = Visualizer(plots_dir)
        os.makedirs(results_dir, exist_ok=True)
    
    def calculate_metrics(self, original_data, imputed_data, mask, scaler, n_features):
        """计算评估指标（在原始尺度上）"""
        # 将数据重塑为2D以便逆标准化
        original_flat = original_data.reshape(-1, n_features)
        imputed_flat = imputed_data.reshape(-1, n_features)
        mask_flat = mask.reshape(-1, n_features)
        
        # 逆标准化到原始尺度
        original_orig_scale = scaler.inverse_transform(original_flat)
        imputed_orig_scale = scaler.inverse_transform(imputed_flat)
        
        # 只计算被mask的位置（即实际缺失值的位置）
        # mask_flat为True表示该位置有原始数据，False表示缺失
        # 我们需要计算的是：在原始数据缺失的位置，插补值与真实值的差异
        # 但这里我们计算的是：在原始数据存在的位置，插补值与原始值的差异（用于验证）
        # 或者：在mask为False的位置（缺失位置），但我们没有真实值，所以用mask为True的位置验证
        
        # 使用mask为True的位置进行验证（这些位置有原始数据）
        valid_mask = mask_flat.any(axis=1)  # 至少有一个特征有值
        
        if valid_mask.sum() > 0:
            original_valid = original_orig_scale[valid_mask]
            imputed_valid = imputed_orig_scale[valid_mask]
            
            # 计算每个特征的指标
            mse_values = []
            mae_values = []
            
            for i in range(n_features):
                feature_mask = mask_flat[valid_mask, i]
                if feature_mask.sum() > 0:
                    orig_feature = original_valid[feature_mask, i]
                    imputed_feature = imputed_valid[feature_mask, i]
                    
                    mse_feature = np.mean((orig_feature - imputed_feature)**2)
                    mae_feature = np.mean(np.abs(orig_feature - imputed_feature))
                    
                    mse_values.append(mse_feature)
                    mae_values.append(mae_feature)
            
            if mse_values:
                mse = np.mean(mse_values)
                mae = np.mean(mae_values)
                rmse = np.sqrt(mse)
            else:
                mse = mae = rmse = 0.0
        else:
            mse = mae = rmse = 0.0
        
        return {
            'MSE': float(mse),
            'MAE': float(mae),
            'RMSE': float(rmse)
        }
    
    def save_results(self, imputed_data, dataset):
        """保存插补结果和评估指标，并生成可视化"""
        print("保存结果和生成可视化...")
        
        metrics_collection = {}
        
        for name, data in imputed_data.items():
            # 保存基本结果
            reshaped_data = data.reshape(-1, dataset['n_features'])
            original_scale_data = dataset['scaler'].inverse_transform(reshaped_data)
            
            df = pd.DataFrame(
                original_scale_data, 
                columns=dataset['feature_columns']
            )
            
            output_path = os.path.join(self.results_dir, f"{name}.csv")
            df.to_csv(output_path, index=False)
            
            # 生成数据分布图
            self.visualizer.plot_data_distribution(
                df, dataset['feature_columns'], 
                f'{name}_after_imputation'
            )
            
            # 生成相关性矩阵
            self.visualizer.plot_correlation_matrix(df, name)
            
            # 生成箱线图
            self.visualizer.plot_boxplot(df, dataset['feature_columns'], name)
            
            # 计算和保存评估指标
            if name != 'test_imputed':
                set_type = name.split('_')[0]
                original_data = dataset[f'{set_type}_X_ori']
                mask = dataset['mask'][set_type]
                
                metrics = self.calculate_metrics(
                    original_data, data, mask, 
                    dataset['scaler'], 
                    dataset['n_features']
                )
                metrics_collection[set_type] = metrics
                
                metrics_df = pd.DataFrame(list(metrics.items()), 
                                        columns=['metric', 'value'])
                metrics_path = os.path.join(self.results_dir, f"{name}_metrics.csv")
                metrics_df.to_csv(metrics_path, index=False, float_format='%.6f')
                print(f"评估指标已保存: {metrics_path}")
                print(f"  MSE: {metrics['MSE']:.6f}")
                print(f"  MAE: {metrics['MAE']:.6f}")
                print(f"  RMSE: {metrics['RMSE']:.6f}")
                
                # 生成插补效果对比图
                self.visualizer.plot_imputation_comparison(
                    original_data, data, mask,
                    dataset['feature_columns'],
                    range(min(3, len(original_data)))
                )
        
        # 生成评估指标对比图
        if metrics_collection:
            self.visualizer.plot_metrics_comparison(
                metrics_collection, 'imputation'
            )

class AirQualityDataProcessor:
    """空气质量数据处理器"""
    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        
    def process_daily_data(self, file_path):
        """处理单日数据文件"""
        try:
            df = pd.read_csv(file_path)
            # 选择数值特征列
            feature_cols = ['PM2.5', 'PM10', 'AQI']
            if all(col in df.columns for col in feature_cols):
                # 按站点分组处理
                processed_data = df[['station_id'] + feature_cols].copy()
                return processed_data
            else:
                print(f"警告: 文件 {file_path} 缺少必要的特征列")
                return None
        except Exception as e:
            print(f"处理文件 {file_path} 时出错: {str(e)}")
            return None
    
    def process_dataset(self, data_list):
        """处理数据集，准备用于插补"""
        try:
            # 合并所有数据
            all_data = pd.concat(data_list, ignore_index=True)
            
            # 获取特征列
            feature_columns = ['PM2.5', 'PM10', 'AQI']
            
            # 按站点分组，构建时间序列
            stations = all_data['station_id'].unique()
            n_stations = len(stations)
            
            # 计算时间步数（假设每天24小时，多个日期）
            n_days = len(data_list)
            n_steps = n_days * 24  # 每天24小时
            
            # 初始化数据数组
            X_list = []
            station_ids = []
            
            for station in stations:
                station_data = all_data[all_data['station_id'] == station].copy()
                station_data = station_data.sort_values(['date', 'timestamp'])
                
                # 提取特征值
                features = station_data[feature_columns].values
                
                # 如果数据不足，进行填充
                if len(features) < n_steps:
                    # 使用NaN填充
                    padding = np.full((n_steps - len(features), len(feature_columns)), np.nan)
                    features = np.vstack([features, padding])
                elif len(features) > n_steps:
                    features = features[:n_steps]
                
                X_list.append(features)
                station_ids.append(station)
            
            # 转换为numpy数组: (n_stations, n_steps, n_features)
            X = np.array(X_list, dtype=np.float32)
            n_features = len(feature_columns)
            
            # 创建缺失值掩码
            mask = ~np.isnan(X)
            
            # 标准化数据（只对非缺失值）
            X_flat = X.reshape(-1, n_features)
            mask_flat = mask.reshape(-1, n_features)
            
            # 对每个特征分别标准化
            X_scaled = X_flat.copy()
            for i in range(n_features):
                valid_values = X_flat[mask_flat[:, i], i]
                if len(valid_values) > 0:
                    self.scaler.fit(valid_values.reshape(-1, 1))
                    X_scaled[mask_flat[:, i], i] = self.scaler.transform(
                        valid_values.reshape(-1, 1)
                    ).flatten()
            
            X_scaled = X_scaled.reshape(n_stations, n_steps, n_features)
            
            # 分割数据集
            train_size = int(0.7 * n_stations)
            val_size = int(0.15 * n_stations)
            
            train_X = X_scaled[:train_size]
            val_X = X_scaled[train_size:train_size + val_size]
            test_X = X_scaled[train_size + val_size:]
            
            train_mask = mask[:train_size]
            val_mask = mask[train_size:train_size + val_size]
            test_mask = mask[train_size + val_size:]
            
            # 创建缺失值用于训练（随机mask一些值）
            train_X_intact = train_X.copy()
            train_X_missing = train_X.copy()
            train_missing_mask = train_mask.copy()
            
            # 随机mask 20%的数据用于训练
            random_mask = np.random.random(train_X.shape) < 0.2
            train_X_missing[random_mask & train_mask] = np.nan
            train_missing_mask = ~np.isnan(train_X_missing) & train_mask
            
            return {
                'n_steps': n_steps,
                'n_features': n_features,
                'feature_columns': feature_columns,
                'scaler': self.scaler,
                'train_X': train_X_missing,
                'train_X_ori': train_X_intact,
                'val_X': val_X,
                'val_X_ori': val_X,
                'test_X': test_X,
                'mask': {
                    'train': train_missing_mask,
                    'val': val_mask,
                    'test': test_mask
                }
            }
        except Exception as e:
            print(f"处理数据集时出错: {str(e)}")
            import traceback
            traceback.print_exc()
            return None

class AirQualityImputer:
    """空气质量数据插补器"""
    def __init__(self, n_steps, n_features, n_epochs=10):
        self.n_steps = n_steps
        self.n_features = n_features
        self.n_epochs = n_epochs
        self.model = None
        
    def perform_imputation(self, dataset):
        """执行插补"""
        try:
            from pypots.imputation import SAITS
            
            # 准备训练数据
            train_X = dataset['train_X']
            train_mask = dataset['mask']['train']
            
            # 初始化模型
            self.model = SAITS(
                n_steps=self.n_steps,
                n_features=self.n_features,
                n_layers=2,
                d_model=256,
                d_inner=128,
                n_heads=4,
                d_k=64,
                d_v=64,
                dropout=0.1,
                epochs=self.n_epochs,
                batch_size=32,
                device='cpu'
            )
            
            print("开始训练插补模型...")
            # 训练模型
            self.model.fit(train_set=(train_X, train_mask))
            
            print("开始执行插补...")
            # 对训练集、验证集、测试集进行插补
            train_imputed = self.model.impute(
                X=dataset['train_X'],
                mask=dataset['mask']['train']
            )
            
            val_imputed = self.model.impute(
                X=dataset['val_X'],
                mask=dataset['mask']['val']
            )
            
            test_imputed = self.model.impute(
                X=dataset['test_X'],
                mask=dataset['mask']['test']
            )
            
            return {
                'train_imputed': train_imputed,
                'val_imputed': val_imputed,
                'test_imputed': test_imputed
            }
        except Exception as e:
            print(f"插补过程中出错: {str(e)}")
            import traceback
            traceback.print_exc()
            return None

def main():
    """主函数"""
    try:
        print("开始空气质量数据时间序列插补...")
        
        # 创建所需目录
        os.makedirs(PLOTS_DIR, exist_ok=True)
        
        # 创建处理器实例
        processor = AirQualityDataProcessor()
        
        # 获取particulate数据文件
        particulate_files = [f for f in os.listdir(DATA_DIR) 
                           if f.endswith('_particulate.csv')]
        print(f"找到 {len(particulate_files)} 个particulate数据文件")
        
        # 处理particulate数据
        particulate_data = []
        for file_name in particulate_files:
            file_path = os.path.join(DATA_DIR, file_name)
            df = processor.process_daily_data(file_path)
            if df is not None:
                particulate_data.append(df)
        
        if not particulate_data:
            raise ValueError("未找到有效的particulate数据文件！")
        
        # 处理数据集
        dataset = processor.process_dataset(particulate_data)
        
        if dataset is not None:
            # 创建插补器
            imputer = AirQualityImputer(
                n_steps=dataset['n_steps'],
                n_features=dataset['n_features']
            )
            
            # 执行插补
            imputed_results = imputer.perform_imputation(dataset)
            
            # 保存结果和生成可视化
            saver = ResultsSaver(RESULTS_DIR, PLOTS_DIR)
            saver.save_results(imputed_results, dataset)
            
            print(f"所有处理完成。可视化结果保存在: {PLOTS_DIR}")
        else:
            print("数据处理失败")
        
    except Exception as e:
        print(f"错误: {str(e)}")
        raise

if __name__ == "__main__":
    main()

2025-05-18 19:18:18 [INFO]: Using the given device: cpu
2025-05-18 19:18:18 [WARNING]: ‼️ saving_path not given. Model files and tensorboard file will not be saved.
2025-05-18 19:18:18 [INFO]: Using customized MAE as the training loss function.
2025-05-18 19:18:18 [INFO]: Using customized MSE as the validation metric function.
2025-05-18 19:18:18 [INFO]: SAITS initialized with the given hyperparameters, the number of trainable parameters: 663,398
2025-05-18 19:18:18 [INFO]: Epoch 001 - training loss (MAE): 1.6877, validation MSE: 1.1098


开始空气质量数据时间序列插补...
找到 31 个particulate数据文件
成功处理文件: 20250101_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250102_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250103_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250104_particulate.csv
数据形状: (735, 6)
成功处理文件: 20250105_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250106_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250107_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250108_particulate.csv
数据形状: (805, 6)
成功处理文件: 20250109_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250110_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250111_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250112_particulate.csv
数据形状: (805, 6)
成功处理文件: 20250113_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250114_particulate.csv
数据形状: (805, 6)
成功处理文件: 20250115_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250116_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250117_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250118_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250119_particulate.csv
数据形状: (840, 6)
成功处理文件: 20250120_particulate.csv
数据形状: (840, 6)

2025-05-18 19:18:19 [INFO]: Epoch 002 - training loss (MAE): 1.6753, validation MSE: 0.8921
2025-05-18 19:18:19 [INFO]: Epoch 003 - training loss (MAE): 1.5580, validation MSE: 0.7251
2025-05-18 19:18:19 [INFO]: Epoch 004 - training loss (MAE): 1.3575, validation MSE: 0.6424
2025-05-18 19:18:19 [INFO]: Epoch 005 - training loss (MAE): 1.3387, validation MSE: 0.5658
2025-05-18 19:18:19 [INFO]: Epoch 006 - training loss (MAE): 1.2647, validation MSE: 0.4672
2025-05-18 19:18:19 [INFO]: Epoch 007 - training loss (MAE): 1.1665, validation MSE: 0.4087
2025-05-18 19:18:19 [INFO]: Epoch 008 - training loss (MAE): 1.0728, validation MSE: 0.3868
2025-05-18 19:18:19 [INFO]: Epoch 009 - training loss (MAE): 1.0087, validation MSE: 0.3575
2025-05-18 19:18:19 [INFO]: Epoch 010 - training loss (MAE): 1.0593, validation MSE: 0.2915
2025-05-18 19:18:19 [INFO]: Finished training. The best model is from epoch#10.


执行数据插补...
保存结果和生成可视化...
所有处理完成。可视化结果保存在: C:\Users\hyhyz\pipotsenvironmentallaw\results\plots


<Figure size 1000x600 with 0 Axes>